=============================================================================
**PHASE 3 — NOTEBOOK 3: MODEL A — GAUSSIAN PROCESS BASELINE**  
=============================================================================
**RUN AFTER: 02_temperature_and_ood.py**  

**WHAT THIS NOTEBOOK DOES:**  
  Builds Model A — the pure ML baseline for the ablation table.
  No neural network involved. Hand-crafted features only.

  Pipeline:
    Image → GLCM + LBP + HSV features → PCA(50) → GP Classifier
                                               ↘
                                         Isolation Forest (OOD)

  WHY this is the right ML baseline:
  - Gaussian Process naturally outputs calibrated probabilities
  - Gives epistemic uncertainty via predictive variance
  - Isolation Forest detects anomalies in feature space
  - All three together form a complete ML uncertainty system

  WHY this will score lower than Model B:
  - Hand-crafted features are a lossy representation of images
  - GP has O(n³) complexity → must subsample training data
  - No transfer learning from ImageNet
  - This is expected — the point is to SHOW the improvement

  Model A results feed directly into the ablation table.
  Without it you cannot claim Model B or C "improve upon" anything.

**ESTIMATED TIME ON T4: ~25-35 minutes**  
  Feature extraction: ~15 minutes (CPU, ~7000 images)
  GP training:        ~10 minutes (2000 samples, n³ complexity)
  Evaluation:         ~5 minutes

**SAVES:**  
  outputs/model_a_features_train.npz
  outputs/model_a_features_test.npz
  outputs/model_a_gp.pkl
  outputs/model_a_isolation_forest.pkl
  outputs/model_a_results.json
=============================================================================

In [1]:
import os, sys, json, time, pickle
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = ".."
sys.path.insert(0, PROJECT_ROOT)

METADATA_CSV = "../../DL/dataset/HAM10000_metadata.csv"
IMAGES_DIR   = "../../DL/dataset/images"
OUTPUTS_DIR  = "../outputs"

from data.dataset import load_metadata, split_dataset, get_dataloaders

# ── sklearn imports ───────────────────────────────────────────────────
from sklearn.gaussian_process  import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.ensemble          import IsolationForest
from sklearn.decomposition     import PCA
from sklearn.preprocessing     import StandardScaler
from sklearn.model_selection   import StratifiedShuffleSplit
from sklearn.metrics           import (f1_score, roc_auc_score,
                                        confusion_matrix)
from PIL                       import Image
from skimage.feature           import graycomatrix, graycoprops
from skimage.feature           import local_binary_pattern
import colorsys

print("=" * 55)
print("MODEL A — GAUSSIAN PROCESS BASELINE")
print("=" * 55)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("Note: Feature extraction runs on CPU (image processing)")

# ── Data ──────────────────────────────────────────────────────────────
print("\nLoading data...")
df = load_metadata(METADATA_CSV, IMAGES_DIR)
df_train, df_val, df_test = split_dataset(df)
print(f"  Train: {len(df_train):,} | Test: {len(df_test):,}")

IDX_TO_CLASS = {0:'nv',1:'mel',2:'bkl',3:'bcc',4:'akiec',5:'vasc',6:'df'}

MODEL A — GAUSSIAN PROCESS BASELINE
Device: cpu
Note: Feature extraction runs on CPU (image processing)

Loading data...
STEP 1B: Loading HAM10000 Metadata
  Total records in CSV: 10015
  Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'dataset']
  Unique lesions (lesion_id): 7470
  Unique images (image_id):   10015
  All 10015 images found on disk. ✅

  Class distribution:
    akiec  (Actinic Keratosis        ):   327 (3.3%)
    bcc    (Basal Cell Carcinoma     ):   514 (5.1%)
    bkl    (Benign Keratosis         ):  1099 (11.0%)
    df     (Dermatofibroma           ):   115 (1.1%)
    mel    (Melanoma                 ):  1113 (11.1%)
    nv     (Melanocytic Nevus        ):  6705 (66.9%)
    vasc   (Vascular Lesion          ):   142 (1.4%)

STEP 1C: Group-Based Train/Val/Test Split
WHY: Splitting by lesion_id prevents same lesion in train+test
  Train:  6959 images | 5228 unique lesions
  Val:    1529 images | 1121 unique lesions
  Test:   1527 images

### STEP 1: HAND-CRAFTED FEATURE EXTRACTION

In [2]:
# Same feature set as Phase 2 step_07_ablation.py.
# Extracting here on T4 is faster (more CPU cores).
#
# Feature vector breakdown (total ~98 features):
#   GLCM properties × 5 props × 8 (2 distances × 4 angles) = 40
#   LBP histogram: 26 values
#   HSV histograms: H(16) + S(8) + V(8) = 32
#   RGB statistics: mean + std × 3 channels = 6
#   Total: 104 values
#
# WHY these features:
#   GLCM: captures texture co-occurrence — melanoma has irregular texture
#   LBP:  local pattern descriptor — robust to illumination changes
#   HSV:  color in perceptual space — melanoma has specific pigmentation
#   RGB stats: global color summary — fast, complementary to HSV

In [3]:
print("\n" + "=" * 55)
print("STEP 1: Hand-Crafted Feature Extraction")
print("=" * 55)

def extract_features(image_path: str, size: int = 64) -> np.ndarray:
    """
    Extract GLCM + LBP + HSV + RGB features from one image.
    Resize to 64×64 first — enough for texture features, much faster.
    """
    try:
        img_rgb  = np.array(
            Image.open(image_path).convert('RGB').resize((size, size))
        )
        img_gray = np.array(
            Image.open(image_path).convert('L').resize((size, size))
        )
    except Exception:
        return np.zeros(104, dtype=np.float32)

    feats = []

    # GLCM texture features
    try:
        glcm = graycomatrix(
            img_gray, distances=[1, 3],
            angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
            levels=256, symmetric=True, normed=True
        )
        for prop in ['contrast', 'correlation', 'energy',
                     'homogeneity', 'ASM']:
            feats.extend(graycoprops(glcm, prop).flatten().tolist())
    except Exception:
        feats.extend([0.0] * 40)

    # LBP histogram
    try:
        lbp      = local_binary_pattern(img_gray, P=24, R=3, method='uniform')
        lbp_hist, _ = np.histogram(lbp, bins=26, range=(0,26), density=True)
        feats.extend(lbp_hist.tolist())
    except Exception:
        feats.extend([0.0] * 26)

    # HSV color histograms
    try:
        img_f  = img_rgb.astype(float) / 255.0
        h_vals, s_vals, v_vals = [], [], []
        for row in img_f:
            for px in row:
                h, s, v = colorsys.rgb_to_hsv(px[0], px[1], px[2])
                h_vals.append(h); s_vals.append(s); v_vals.append(v)
        h_hist, _ = np.histogram(h_vals, bins=16, range=(0,1), density=True)
        s_hist, _ = np.histogram(s_vals, bins=8,  range=(0,1), density=True)
        v_hist, _ = np.histogram(v_vals, bins=8,  range=(0,1), density=True)
        feats.extend(h_hist.tolist())
        feats.extend(s_hist.tolist())
        feats.extend(v_hist.tolist())
    except Exception:
        feats.extend([0.0] * 32)

    # RGB statistics
    for c in range(3):
        feats.append(img_rgb[:,:,c].mean() / 255.0)
        feats.append(img_rgb[:,:,c].std()  / 255.0)

    return np.array(feats, dtype=np.float32)


def extract_batch(df_split, desc='', n_limit=None):
    """Extract features for all images in a DataFrame split."""
    if n_limit:
        df_split = df_split.sample(
            min(n_limit, len(df_split)), random_state=42
        )
    X, y = [], []
    total = len(df_split)
    t0 = time.time()

    for i, (_, row) in enumerate(df_split.iterrows()):
        feats = extract_features(row['image_path'])
        X.append(feats)
        y.append(row['label'])
        if (i+1) % 500 == 0:
            elapsed = time.time() - t0
            rate    = (i+1) / elapsed
            eta     = (total - i - 1) / rate
            print(f"  {desc}: {i+1}/{total} images "
                  f"({rate:.0f} img/s, ETA: {eta:.0f}s)")

    return np.array(X), np.array(y)


# Check if features already extracted (resume-friendly)
train_feat_path = f"{OUTPUTS_DIR}/model_a_features_train.npz"
test_feat_path  = f"{OUTPUTS_DIR}/model_a_features_test.npz"

if os.path.exists(train_feat_path) and os.path.exists(test_feat_path):
    print("  Loading cached features...")
    train_data = np.load(train_feat_path)
    X_train_all, y_train_all = train_data['X'], train_data['y']
    test_data  = np.load(test_feat_path)
    X_test,  y_test          = test_data['X'],  test_data['y']
    print(f"  Train features: {X_train_all.shape}")
    print(f"  Test features:  {X_test.shape}")
else:
    print("  Extracting training features...")
    t0 = time.time()
    X_train_all, y_train_all = extract_batch(df_train, desc='Train')
    print(f"  Train done in {time.time()-t0:.0f}s — shape: {X_train_all.shape}")

    print("\n  Extracting test features...")
    t0 = time.time()
    X_test, y_test = extract_batch(df_test, desc='Test')
    print(f"  Test done in {time.time()-t0:.0f}s — shape: {X_test.shape}")

    # Cache for future runs
    np.savez(train_feat_path, X=X_train_all, y=y_train_all)
    np.savez(test_feat_path,  X=X_test,      y=y_test)
    print(f"  Features cached to {OUTPUTS_DIR}/")

n_features = X_train_all.shape[1]
print(f"\n  Feature vector dimension: {n_features}")


STEP 1: Hand-Crafted Feature Extraction
  Extracting training features...


  Train: 500/6959 images (36 img/s, ETA: 178s)


  Train: 1000/6959 images (36 img/s, ETA: 165s)


  Train: 1500/6959 images (37 img/s, ETA: 148s)


  Train: 2000/6959 images (38 img/s, ETA: 132s)


...

  Train done in 184s — shape: (6959, 104)

  Extracting test features...


  Test: 500/1527 images (35 img/s, ETA: 29s)


  Test: 1000/1527 images (37 img/s, ETA: 14s)


  Test: 1500/1527 images (37 img/s, ETA: 1s)


  Test done in 42s — shape: (1527, 104)
  Features cached to ../outputs/

  Feature vector dimension: 104


### STEP 2: PREPROCESSING — SCALE + PCA

In [4]:
# WHY scale: features have different ranges
#   GLCM contrast: [0, 500], LBP: [0, 1], HSV: [0, 1]
#   StandardScaler brings all to zero mean, unit variance.
#
# WHY PCA(50): GP complexity is O(n³) in training samples.
#   With 2000 samples and 104 features, the kernel matrix is 2000×2000.
#   High-dimensional features make the RBF kernel nearly constant
#   (curse of dimensionality). PCA to 50 dimensions resolves this.
#   50 components typically captures >90% of variance.

In [5]:
print("\n" + "=" * 55)
print("STEP 2: Preprocessing — Scale + PCA")
print("=" * 55)

scaler = StandardScaler()
pca    = PCA(n_components=50, random_state=42)

X_train_scaled = scaler.fit_transform(X_train_all)
X_train_pca    = pca.fit_transform(X_train_scaled)
var_retained   = pca.explained_variance_ratio_.sum()

X_test_scaled  = scaler.transform(X_test)
X_test_pca     = pca.transform(X_test_scaled)

print(f"  Scaled to zero mean, unit variance")
print(f"  PCA: {n_features} → 50 dimensions")
print(f"  Variance retained: {var_retained:.1%}")


STEP 2: Preprocessing — Scale + PCA
  Scaled to zero mean, unit variance
  PCA: 104 → 50 dimensions
  Variance retained: 98.2%


### STEP 3: SUBSAMPLE FOR GP TRAINING

In [6]:
# GP cannot train on all 7000 samples — O(n³) is too slow.
# 2000 stratified samples is the practical limit.
# WHY stratified: ensures rare classes are represented.

In [7]:
N_TRAIN_GP = 2000
print(f"\n  Subsampling {N_TRAIN_GP} stratified samples for GP training...")
print(f"  (GP has O(n³) complexity — 7000 samples would take hours)")

sss = StratifiedShuffleSplit(
    n_splits=1, train_size=N_TRAIN_GP, random_state=42
)
train_idx, _ = next(sss.split(X_train_pca, y_train_all))
X_gp   = X_train_pca[train_idx]
y_gp   = y_train_all[train_idx]

print(f"  GP training set: {X_gp.shape}")
from collections import Counter
print(f"  Class distribution:")
for cls_idx, count in sorted(Counter(y_gp).items()):
    print(f"    {IDX_TO_CLASS[cls_idx]}: {count}")


  Subsampling 2000 stratified samples for GP training...
  (GP has O(n³) complexity — 7000 samples would take hours)
  GP training set: (2000, 50)
  Class distribution:
    nv: 1340
    mel: 222
    bkl: 221
    bcc: 100
    akiec: 64
    vasc: 27
    df: 26


### STEP 4: ISOLATION FOREST — OOD DETECTION

In [8]:
# Fits on the same 2000 GP training samples.
# At inference, scores new samples — negative = anomaly.
# WHY isolation forest: it doesn't assume a specific distribution.
# Trees isolate points: anomalies are isolated faster (fewer splits).

In [9]:
print("\n" + "=" * 55)
print("STEP 4: Isolation Forest — OOD Detection")
print("=" * 55)

iso_forest = IsolationForest(
    n_estimators  = 200,
    contamination = 0.05,  # assume 5% anomaly rate
    random_state  = 42,
    n_jobs        = -1
)
t0 = time.time()
iso_forest.fit(X_gp)
print(f"  Trained in {time.time()-t0:.1f}s")
print(f"  Contamination: 5% (1 in 20 real-world clinical images)")


STEP 4: Isolation Forest — OOD Detection


  Trained in 0.2s
  Contamination: 5% (1 in 20 real-world clinical images)


### STEP 5: GAUSSIAN PROCESS CLASSIFIER

In [10]:
# RBF kernel + WhiteKernel noise.
# WHY RBF: k(x,x') = exp(-||x-x'||²/2l²)
#   Similar feature vectors → high similarity → similar prediction.
#   This is the correct prior for our feature space.
# WHY WhiteKernel: models observation noise, prevents overfitting.
# WHY multi_class='one_vs_rest': 7-class GP is too expensive.
#   OVR trains 7 binary GPs — each class vs all others.

In [11]:
print("\n" + "=" * 55)
print("STEP 5: Gaussian Process Classifier")
print("=" * 55)
print(f"  Kernel: 1.0 × RBF + WhiteKernel")
print(f"  Multi-class: one_vs_rest (7 binary GPs)")
print(f"  Training on {N_TRAIN_GP} samples...")

kernel = 1.0 * RBF(length_scale=1.0) + WhiteKernel(noise_level=0.1)
gp     = GaussianProcessClassifier(
    kernel                = kernel,
    multi_class           = 'one_vs_rest',
    n_restarts_optimizer  = 2,
    random_state          = 42,
    n_jobs                = -1
)

t0 = time.time()
gp.fit(X_gp, y_gp)
gp_train_time = time.time() - t0
print(f"  Trained in {gp_train_time:.1f}s")
print(f"  Optimised kernel: {gp.kernel_}")


STEP 5: Gaussian Process Classifier
  Kernel: 1.0 × RBF + WhiteKernel
  Multi-class: one_vs_rest (7 binary GPs)
  Training on 2000 samples...


/Users/princesahoo/CODE/Academic Projects/Robust-Medical-Vision/DL/.venv/lib/python3.9/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


/Users/princesahoo/CODE/Academic Projects/Robust-Medical-Vision/DL/.venv/lib/python3.9/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


/Users/princesahoo/CODE/Academic Projects/Robust-Medical-Vision/DL/.venv/lib/python3.9/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


/Users/princesahoo/CODE/Academic Projects/Robust-Medical-Vision/DL/.venv/lib/python3.9/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


/Users/princesahoo/CODE/Academic Projects/Robust-Medical-Vision/DL/.venv/lib/python3.9/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


/Users/princesahoo/CODE/Academic Projects/Robust-Medical-Vision/DL/.venv/lib/python3.9/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


  Trained in 358.8s
  Optimised kernel: CompoundKernel(2.66, 2.35, -11.5, 3.25, 3.01, -11.5, 2.91, 2.78, -11.5, 3.31, 3.07, -11.5, 5.93, 4.26, -4.38, 4.91, 4.04, -11.5, 5.34, 4.11, -11.5)


In [12]:
# STEP 6: EVALUATE MODEL A ON TEST SET

In [13]:
print("\n" + "=" * 55)
print("STEP 6: Model A Evaluation on Test Set")
print("=" * 55)

t0         = time.time()
y_pred_gp  = gp.predict(X_test_pca)
y_proba_gp = gp.predict_proba(X_test_pca)
ood_flags  = iso_forest.predict(X_test_pca)   # -1=anomaly, 1=normal
eval_time  = time.time() - t0

# Metrics
f1_macro    = f1_score(y_test, y_pred_gp, average='macro', zero_division=0)
f1_per_cls  = f1_score(y_test, y_pred_gp, average=None,   zero_division=0)
try:
    auroc = roc_auc_score(
        y_test, y_proba_gp, multi_class='ovr', average='macro'
    )
except Exception:
    auroc = 0.0

ood_rate = (ood_flags == -1).mean() * 100

print(f"\n  Model A (GP + Isolation Forest) Results:")
print(f"    F1 (macro): {f1_macro:.4f}")
print(f"    AUROC:      {auroc:.4f}")
print(f"    OOD rate:   {ood_rate:.1f}% flagged as anomalous")
print(f"    Eval time:  {eval_time:.1f}s on {len(y_test)} samples")

print(f"\n  Per-class F1:")
class_codes = ['nv','mel','bkl','bcc','akiec','vasc','df']
for cls, f1_v in zip(class_codes, f1_per_cls):
    bar = '█' * int(f1_v * 25)
    print(f"    {cls:<8}: {f1_v:.3f}  {bar}")

# GP uncertainty: use probability spread as uncertainty measure
# Higher entropy of predicted probabilities = higher uncertainty
gp_entropy = -(y_proba_gp * np.log(y_proba_gp + 1e-8)).sum(axis=1)

correct = (y_pred_gp == y_test)
unc_correct = gp_entropy[correct].mean()
unc_wrong   = gp_entropy[~correct].mean()
print(f"\n  GP Uncertainty Validation:")
print(f"    Mean entropy (correct):   {unc_correct:.4f}")
print(f"    Mean entropy (wrong):     {unc_wrong:.4f}")
if unc_wrong > unc_correct:
    print(f"    ✅ Higher entropy on wrong predictions — GP is self-aware")
else:
    print(f"    ⚠️  GP uncertainty doesn't track difficulty well")


STEP 6: Model A Evaluation on Test Set



  Model A (GP + Isolation Forest) Results:
    F1 (macro): 0.3488
    AUROC:      0.8616
    OOD rate:   5.0% flagged as anomalous
    Eval time:  3.3s on 1527 samples

  Per-class F1:
    nv      : 0.857  █████████████████████
    mel     : 0.401  ██████████
    bkl     : 0.389  █████████
    bcc     : 0.370  █████████
    akiec   : 0.424  ██████████
    vasc    : 0.000  
    df      : 0.000  

  GP Uncertainty Validation:
    Mean entropy (correct):   0.7744
    Mean entropy (wrong):     1.2806
    ✅ Higher entropy on wrong predictions — GP is self-aware


In [14]:
# STEP 7: SAVE MODEL A ARTIFACTS

In [15]:
artifacts = {
    'scaler':     scaler,
    'pca':        pca,
    'gp':         gp,
    'iso_forest': iso_forest,
}
with open(f"{OUTPUTS_DIR}/model_a_gp.pkl", 'wb') as f:
    pickle.dump(artifacts, f)
print(f"\n  Model A artifacts saved: {OUTPUTS_DIR}/model_a_gp.pkl")

results_a = {
    'f1_macro':     float(f1_macro),
    'auroc':        float(auroc),
    'f1_per_class': [float(x) for x in f1_per_cls],
    'ood_rate':     float(ood_rate),
    'n_train_gp':   N_TRAIN_GP,
    'n_features':   n_features,
    'pca_components': 50,
    'variance_retained': float(var_retained),
    'unc_correct':  float(unc_correct),
    'unc_wrong':    float(unc_wrong),
    'description':  'GLCM+LBP+HSV → StandardScaler → PCA(50) → GP (OVR) + IsoForest',
}
with open(f"{OUTPUTS_DIR}/model_a_results.json", 'w') as f:
    json.dump(results_a, f, indent=2)


print("\n" + "=" * 55)
print("MODEL A COMPLETE")
print("=" * 55)
print(f"""
  Results:
    F1 (macro): {f1_macro:.4f}  ← ML-only ceiling
    AUROC:      {auroc:.4f}
    OOD rate:   {ood_rate:.1f}%

  Files saved:
    {OUTPUTS_DIR}/model_a_gp.pkl
    {OUTPUTS_DIR}/model_a_features_train.npz
    {OUTPUTS_DIR}/model_a_features_test.npz
    {OUTPUTS_DIR}/model_a_results.json

  EXPECTED: Model B F1 should be ~0.10-0.15 higher than this.
            If not, check that Model B training completed correctly.

→ NEXT: Run 04_conformal_prediction.py
""")


  Model A artifacts saved: ../outputs/model_a_gp.pkl

MODEL A COMPLETE

  Results:
    F1 (macro): 0.3488  ← ML-only ceiling
    AUROC:      0.8616
    OOD rate:   5.0%

  Files saved:
    ../outputs/model_a_gp.pkl
    ../outputs/model_a_features_train.npz
    ../outputs/model_a_features_test.npz
    ../outputs/model_a_results.json

  EXPECTED: Model B F1 should be ~0.10-0.15 higher than this.
            If not, check that Model B training completed correctly.

→ NEXT: Run 04_conformal_prediction.py

